# C3 — Z-Score Path with Trade Markers

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
for ax, (wk, wm) in zip(axes, WINDOWS_META.items()):
    rk = wm['result_key']
    ts = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'timeseries.parquet')
    ts.index = pd.to_datetime(ts.index)
    ts_1min = ts['zscore'].resample('1min').last().dropna()
    ax.plot(ts_1min.index, ts_1min.values, lw=0.6, color='grey', alpha=0.8)

    trades = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'trades.parquet')
    trades['entry_time'] = pd.to_datetime(trades['entry_time'])
    longs  = trades[trades['dir_label'] == 'Long']
    shorts = trades[trades['dir_label'] == 'Short']
    entry_z_long  = trades.loc[longs.index,  'entry_z']
    entry_z_short = trades.loc[shorts.index, 'entry_z']
    ax.scatter(longs['entry_time'],  entry_z_long,  marker='^', color='#2ecc71', s=30, zorder=5, label='Long')
    ax.scatter(shorts['entry_time'], entry_z_short, marker='v', color='#e74c3c', s=30, zorder=5, label='Short')
    ax.axhline(0, color='black', lw=0.5)
    ax.axhline(2.5, color='grey', ls='--', lw=0.5)
    ax.axhline(-2.5, color='grey', ls='--', lw=0.5)
    ax.set_title(f'{wk}: {wm["front"]}→{wm["back"]}')
    ax.set_ylabel('Z-Score')
    ax.legend(fontsize=8)
fig.suptitle('Z-Score Path with Trade Entry Markers — All Windows', fontsize=13)
save_fig(fig, 'C3_zscore_path_with_trades.png')
